# Using a Vision Transformer on CIFAR-100

## Load dataset from Huggingface

In [ ]:
import umap
from renumics import spotlight

import datasets

ds = datasets.load_dataset("cifar100", split="test")
ds = ds.select(range(0, 100))
spotlight.show(ds, analyze=False)

## Compute embedding with vision transformer

In [ ]:
import torch
import transformers

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_name = "Ahmed9275/Vit-Cifar100"
processor = transformers.ViTImageProcessor.from_pretrained(model_name)
cls_model = transformers.ViTForImageClassification.from_pretrained(model_name).to(device)
fe_model = transformers.ViTModel.from_pretrained(model_name).to(device)


def infer(batch):
    images = [image.convert("RGB") for image in batch]
    inputs = processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = cls_model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
        embeddings = fe_model(**inputs).last_hidden_state[:, 0].cpu().numpy()
    preds = probs.argmax(axis=-1)
    return {"prediction": preds, "embedding": embeddings}


features = datasets.Features(
    {
        **ds.features,
        "prediction": ds.features["fine_label"],
        "embedding": datasets.Sequence(feature=datasets.Value("float32"), length=768),
    }
)
ds_enriched = ds.map(infer, input_columns="img", batched=True, batch_size=2, features=features)

# Optional: Load enrichment from Huggingface

In [ ]:
#ds_results = datasets.load_dataset("renumics/spotlight-cifar100-enrichment", split="test")
#ds_enriched = datasets.concatenate_datasets([ds, ds_results], axis=1)

# Visualize with Spotlight

In [ ]:
layout = spotlight.layouts.debug_classification(
    label="fine_label", embedding="embedding", inspect={"img": spotlight.dtypes.image_dtype}
)
spotlight.show(ds_enriched, dtype={"embedding": spotlight.Embedding}, analyze=False, layout=layout)

# Optional: Compute clustering with UMAP

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Convert embeddings to numpy array
embeddings = np.array(ds_enriched['embedding'])

# Apply UMAP to reduce to 2D
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(embeddings)

# Get labels for coloring
labels = ds_enriched['fine_label']
class_names = ds_enriched.features['fine_label'].names

# Create the plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=labels, cmap='tab20', alpha=0.7, s=50)

# Add colorbar
cbar = plt.colorbar(scatter)
cbar.set_label('Class Label')

plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.title('2D UMAP Embedding of CIFAR-100 Features')
plt.grid(True, alpha=0.3)

# Show the plot
plt.tight_layout()
plt.show()

print(f"Original embedding shape: {embeddings.shape}")
print(f"Reduced embedding shape: {embeddings_2d.shape}")
print(f"Number of unique classes: {len(set(labels))}")